# Trend Following Strategy — Documentación del Proyecto
### Ingeniería Financiera F414 — Universidad de San Andrés (2026)

Este notebook documenta el proyecto **en orden de construcción**: desde la intuición económica
y los papers hasta cada módulo de código, su lógica matemática y cómo se conectan entre sí.
Es la guía principal para entender y presentar el proyecto.

---

## Índice
1. [Intuición Económica y Base Bibliográfica](#1)
2. [Arquitectura del Sistema](#2)
3. [Capa de Datos: `src/data/`](#3)
4. [Estrategia TSMOM: `src/strategy/`](#4)
5. [Backtesting Framework: `src/backtest/`](#5)
6. [Tests de Robustez: `src/robustness/`](#6)
7. [Broker IBKR: `src/broker/`](#7)
8. [CLI: `main.py`](#8)
9. [Dashboard: `src/dashboard/app.py`](#9)
10. [Configuración Central: `config.yaml`](#10)
11. [Demo End-to-End](#11)

---
<a id='1'></a>
## 1. Intuición Económica y Base Bibliográfica

### ¿Qué es Time Series Momentum (TSMOM)?

La idea es simple: **los activos que subieron en los últimos 12 meses tienden a seguir subiendo el mes siguiente**, y los que bajaron tienden a seguir bajando. A diferencia del momentum transversal (que compara activos entre sí), el TSMOM evalúa cada activo contra su propio historial.

**Señal:**
$$s_{i,t} = \text{sign}\left(\sum_{k=1}^{12} r_{i,t-k}\right) = \begin{cases} +1 & \text{si el activo subió → COMPRAR} \\ -1 & \text{si el activo bajó → VENDER (short)} \end{cases}$$

### ¿Por qué funciona? (3 teorías)

| Teoría | Mecanismo |
|--------|-----------|
| **Underreaction** | Los inversores reaccionan lento a nueva información → el precio ajusta gradualmente |
| **Herding** | Los managers compran lo que subió por presión de clientes → auto-cumplimiento |
| **Delegated management** | Los hedge funds tienen mandatos que los obligan a seguir tendencias |

### Base Bibliográfica del Proyecto

| Rol | Paper | Contribución al proyecto |
|-----|-------|-------------------------|
| **[L] Principal** | Moskowitz, Ooi & Pedersen (2012) — *Time Series Momentum*, JFE | Diseño de señal, vol-scaling, universo multi-asset |
| **[H] Complementario** | Baz et al. (2015) — *Dissecting Investment Strategies* | Señal continua normalizada (`signal_strength`), multi-timescale |
| **Adicional 1** | Hurst, Ooi & Pedersen (2017) — *A Century of Evidence on Trend-Following*, JPM | Valida que TSMOM funciona en 100+ años y múltiples asset classes |
| **Adicional 2** | Barroso & Santa-Clara (2015) — *Momentum Has Its Moments*, JFE | Escalar por volatilidad reduce crashes — justifica nuestro EWMA vol-scaling |

---
<a id='2'></a>
## 2. Arquitectura del Sistema

El proyecto se divide en capas independientes que se comunican de forma unidireccional:

```
┌───────────────────────────────────────────────────────────────┐
│                        main.py  (CLI — Click)                  │
│  update-data │ backtest │ robustness │ execute │ monitor │ dashboard │
└──────┬────────────────────────────────────────────────────────┘
       │                                           │
  ┌────┴─────────────────┐              ┌──────────┴──────────┐
  │  src/data/           │              │  src/broker/        │
  │  ├─ universe.py      │              │  ├─ ibkr.py         │
  │  └─ loader.py        │              │  ├─ orders.py       │
  └────┬─────────────────┘              │  └─ monitor.py      │
       │                                └─────────────────────┘
  ┌────▼─────────────────┐
  │  src/strategy/       │
  │  ├─ volatility.py    │
  │  ├─ signals.py       │
  │  └─ portfolio.py     │
  └────┬─────────────────┘
       │
  ┌────▼─────────────────┐
  │  src/backtest/       │
  │  ├─ engine.py        │
  │  ├─ costs.py         │
  │  └─ metrics.py       │
  └────┬─────────────────┘
       │
  ┌────┴──────┐  ┌─────────────────────┐
  │robustness/│  │  src/dashboard/     │
  │sensitivity│  │  └─ app.py          │
  │stress_test│  │  (Streamlit, 5 pág.)│
  │ oos       │  └─────────────────────┘
  └───────────┘
```

**Principios de diseño:**
- **Una responsabilidad por módulo**: `strategy/` solo produce señales y pesos, `backtest/` solo simula el portafolio. Ningún módulo conoce la implementación interna de otro.
- **`config.yaml` como única fuente de verdad**: todos los parámetros (lookback, target_vol, costos, fechas) viven ahí. Cambiar un valor cambia el comportamiento de todo el sistema.
- **Sin look-ahead bias**: cada cálculo usa `.shift(1)` donde corresponde — garantiza que en el mes $t$ solo se usa información hasta $t-1$.
- **Separación CLI / lógica**: `main.py` solo orquesta llamadas; toda la lógica vive en `src/`.

---
<a id='3'></a>
## 3. Capa de Datos: `src/data/`

### 3.1 `src/data/universe.py` — El universo de ETFs

**¿Qué hace?** Define el diccionario `ETF_UNIVERSE`: un mapeo `ticker → clase de activo`.

**¿Por qué ETFs y no futuros?**  
Moskowitz et al. (2012) usaron futuros (más líquidos, menores costos de financiamiento). Nosotros usamos ETFs porque:
1. Los datos son gratuitos vía yfinance
2. Son ejecutables directamente desde una cuenta IBKR paper sin acceso a futuros
3. El paper señala explícitamente que los resultados son similares en ETFs

**Las 4 clases de activos** replican la diversificación del paper (equities, bonds, commodities, currencies).

In [ ]:
import sys
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import yaml

plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})

from src.data.universe import ETF_UNIVERSE, ASSET_CLASSES

df_u = pd.DataFrame.from_dict(ETF_UNIVERSE, orient='index', columns=['asset_class'])
print('── ETFs por clase de activo ──')
print(df_u['asset_class'].value_counts().to_string())
print(f'\nTotal: {len(ETF_UNIVERSE)} ETFs')
print()
for ac in ASSET_CLASSES:
    tickers = df_u[df_u['asset_class'] == ac].index.tolist()
    print(f'{ac:12}: {tickers}')

### 3.2 `src/data/loader.py` — Carga y actualización de datos

**Pipeline de datos:**

```
Data/etf_prices.parquet  →  load_etf_prices()  →  precios diarios [días × tickers]
                                  │
                                  ▼  compute_monthly_returns()
                         resample('ME').last()  →  retornos mensuales [meses × tickers]
                                  │
                                  ▼  load_returns(config)
                         filtro de fechas + dropna  →  DataFrame listo para usar
```

**Decisiones de diseño:**
- `resample('ME').last()`: tomamos el último precio del mes (no el promedio) — estándar académico y evita forward-looking bias
- `ffill().bfill()`: ETFs distintos cotizan en distintos días. Rellenar evita que un NaN aislado destruya la señal de todo el mes
- `dropna(thresh=80%)`: descartamos ETFs con más del 20% de meses sin datos — garantiza señales confiables
- `update_prices()` con yfinance: si el parquet tiene datos hasta el 2024, descarga solo desde la última fecha disponible (no re-descarga todo)

In [ ]:
from src.data.loader import load_etf_prices, load_returns

with open('config.yaml') as f:
    config = yaml.safe_load(f)

prices = load_etf_prices(config['data']['path'])
returns = load_returns(config)

print(f'Precios diarios : {prices.shape[1]} ETFs × {prices.shape[0]} días')
print(f'  Rango         : {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'\nRetornos mensual: {returns.shape[1]} ETFs × {returns.shape[0]} meses')
print(f'  Rango         : {returns.index[0].date()} → {returns.index[-1].date()}')
print(f'\nEstadísticas de retornos mensuales (pooled):')
pooled = returns.values.flatten()
pooled = pooled[~pd.isna(pooled)]
print(f'  Media   : {pooled.mean():.4f}  ({pooled.mean()*12:.2%} anualizado)')
print(f'  Std     : {pooled.std():.4f}')
print(f'  Skew    : {pd.Series(pooled).skew():.2f}')
print(f'  Kurtosis: {pd.Series(pooled).kurtosis():.2f}')

---
<a id='4'></a>
## 4. Estrategia TSMOM: `src/strategy/`

Tres archivos que implementan la lógica del paper, en orden de dependencia:

```
volatility.py  →  signals.py  →  portfolio.py
   (σ̂_{i,t})      (s_{i,t})      (w_{i,t})
```

### 4.1 `src/strategy/volatility.py` — Volatilidad ex-ante

**¿Por qué necesitamos la volatilidad?**  
Si ponemos igual capital en USO (vol ~40%) y TLT (vol ~10%), el 95% del riesgo viene de oil.
La solución es **igualar la contribución al riesgo** de cada activo.

**Fórmula EWMA (Moskowitz Eq. 1):**
$$\hat{\sigma}_{i,t}^2 = \frac{\sum_{k=0}^{\infty} (1-\delta)\,\delta^k \, r_{i,t-1-k}^2}{\text{normalización}}$$

Implementado con `pandas.ewm(com=3)` donde `com=3 meses ≈ 60 días de trading` (parámetro del paper).

**El `.shift(1)` es crítico:**  
Sin él, $\hat{\sigma}_{i,t}$ incluiría $r_{i,t}$ (el retorno de este mes), que todavía no conocemos cuando tomamos la decisión. Eso es *look-ahead bias*.

```python
# Con look-ahead bias (INCORRECTO):
ewma_var = (returns ** 2).ewm(com=3).mean()

# Sin look-ahead bias (CORRECTO):
ewma_var = (returns ** 2).ewm(com=3).mean().shift(1)
```

In [ ]:
from src.strategy.volatility import compute_ex_ante_vol

vol = compute_ex_ante_vol(returns, com_months=3)

median_vol = vol.median().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

top15 = median_vol.head(15)
axes[0].barh(range(len(top15)), top15.values * 100, color='steelblue', alpha=0.8)
axes[0].set_yticks(range(len(top15)))
axes[0].set_yticklabels(top15.index)
axes[0].set_xlabel('Volatilidad anualizada mediana (%)')
axes[0].set_title('Top 15 ETFs más volátiles')

for ticker in ['SPY', 'TLT', 'GLD', 'USO']:
    if ticker in vol.columns:
        axes[1].plot(vol.index, vol[ticker] * 100, label=ticker, alpha=0.85)
axes[1].set_title('Volatilidad ex-ante — evolución temporal')
axes[1].set_ylabel('Vol anualizada (%)')
axes[1].legend()

plt.tight_layout(); plt.show()

print('Mediana de vol por clase de activo:')
for ac in ASSET_CLASSES:
    tc = [t for t in vol.columns if ETF_UNIVERSE.get(t) == ac]
    if tc:
        print(f'  {ac:12}: {vol[tc].median().median():.1%}')

### 4.2 `src/strategy/signals.py` — La señal de momentum

**Función 1 — Señal binaria (Moskowitz et al. 2012):**
$$s_{i,t} = \text{sign}\bigl(r_{i,t-12,t-1}\bigr)$$

Solo importa la **dirección** del movimiento. El paper demuestra que agregar la magnitud no mejora la predicción.

**Función 2 — Señal continua (Baz et al. 2015):**
$$z_{i,t} = \text{clip}\!\left(\frac{r_{i,t-12,t-1}}{\sigma(r_{i,t-12,t-1})},\; -3,\; 3\right)$$

Normaliza por la volatilidad del retorno acumulado. Activos con tendencias más fuertes **y más consistentes** reciben posiciones más grandes. Se clipea a ±3 para no concentrar demasiado en outliers.

**Comparación:**
| | Binaria | Continua |
|-|---------|----------|
| Turnover | Bajo (cambia solo cuando invierte) | Mayor |
| Robustez | Alta (menos parámetros) | Moderada |
| Origen | Moskowitz 2012 (paper principal) | Baz 2015 (paper complementario) |

**Ambas están implementadas** y se seleccionan con el flag `--signal binary/strength` en el CLI.

In [ ]:
from src.strategy.signals import compute_tsmom_signal, compute_signal_strength

signal_bin  = compute_tsmom_signal(returns, lookback=12)
signal_cont = compute_signal_strength(returns, lookback=12)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Fracción del tiempo en LONG
pct_long = (signal_bin == 1).mean().sort_values(ascending=False)
axes[0].bar(range(len(pct_long)), pct_long.values, color='steelblue', alpha=0.7, width=1.0)
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=1, label='50%')
axes[0].set_title('% del tiempo LONG por ETF')
axes[0].set_xlabel('ETF (ordenado)')
axes[0].legend()

# Señal binaria — SPY
if 'SPY' in signal_bin.columns:
    s = signal_bin['SPY'].dropna()
    axes[1].step(s.index, s.values, where='post', color='darkorange', linewidth=1)
    axes[1].fill_between(s.index, s.values, 0, alpha=0.2, color='darkorange')
    axes[1].set_title('Señal binaria — SPY')
    axes[1].set_ylabel('+1 LONG / -1 SHORT')
    axes[1].set_ylim(-1.5, 1.5)

# Señal continua — SPY
if 'SPY' in signal_cont.columns:
    sc = signal_cont['SPY'].dropna()
    axes[2].plot(sc.index, sc.values, color='navy', linewidth=1)
    axes[2].axhline(0, color='black', linewidth=0.5)
    axes[2].axhline(1, color='green', linestyle=':', linewidth=0.8)
    axes[2].axhline(-1, color='red', linestyle=':', linewidth=0.8)
    axes[2].set_title('Señal continua (Baz) — SPY')
    axes[2].set_ylabel('Z-score clipado [-3, 3]')

plt.tight_layout(); plt.show()
print(f'Promedio % LONG (todas las observaciones): {(signal_bin==1).mean().mean():.1%}')

### 4.3 `src/strategy/portfolio.py` — Sizing de posiciones

**La fórmula central (Moskowitz Eq. 4):**

$$w_{i,t} = \frac{s_{i,t}}{N_t} \cdot \frac{\sigma_{\text{target}}}{\hat{\sigma}_{i,t}}$$

| Variable | Valor | Rol |
|----------|-------|-----|
| $s_{i,t}$ | +1 o −1 | Dirección |
| $N_t$ | ~60 ETFs | Normaliza por cantidad de activos activos |
| $\sigma_{\text{target}}$ | 10% anual | Volatilidad objetivo del portafolio |
| $\hat{\sigma}_{i,t}$ | Variable | Volatilidad EWMA del activo |

**Ejemplo numérico:**
Con target_vol=10%, N=60, señal=+1:
- USO (vol=40%): w = 1/60 × 10%/40% = **0.42%** → posición pequeña
- TLT (vol=10%): w = 1/60 × 10%/10% = **1.67%** → posición grande
- Ambos contribuyen ~0.17% de volatilidad al portfolio

**`apply_position_constraints()`**: clipea a ±15% para evitar concentración extrema ante outliers de volatilidad muy baja.

In [ ]:
from src.strategy.portfolio import compute_vol_scaled_weights, apply_position_constraints

weights = apply_position_constraints(
    compute_vol_scaled_weights(signal_bin, vol, target_vol=0.10),
    max_weight=0.15
)

last_w = weights.dropna(how='all').iloc[-1].sort_values()
last_date = weights.index[-1].strftime('%Y-%m')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['salmon' if v < 0 else 'steelblue' for v in last_w.values]
axes[0].barh(range(len(last_w)), last_w.values * 100, color=colors, alpha=0.85)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_yticks(range(0, len(last_w), 5))
axes[0].set_yticklabels(last_w.index[::5])
axes[0].set_title(f'Pesos — {last_date}')
axes[0].set_xlabel('Peso (%)')

axes[1].plot(weights.sum(axis=1).index, weights.sum(axis=1).values * 100,
             label='Exposición neta', color='navy')
axes[1].plot(weights.abs().sum(axis=1).index, weights.abs().sum(axis=1).values * 100,
             label='Exposición bruta', color='darkorange', linestyle='--')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Exposición del portafolio (%)')
axes[1].legend()

plt.tight_layout(); plt.show()

# Verificar contribución de riesgo
risk_contrib = (weights.abs() * vol).dropna(how='all').iloc[-1].sort_values(ascending=False)
print(f'Top 5 contribuciones al riesgo (mes {last_date}):')
print((risk_contrib.head(5) * 100).round(3).to_string())

---
<a id='5'></a>
## 5. Backtesting Framework: `src/backtest/`

### 5.1 `src/backtest/costs.py` — Costos de transacción

**El problema:** un backtest sin costos sobreestima el retorno. El TSMOM rebalancea mensualmente: cada mes hay que pagar el spread bid-ask y las comisiones.

**Fórmula:**
$$c_t = \underbrace{\sum_i |\Delta w_{i,t}|}_{\text{turnover}} \cdot \left(\frac{\text{spread}}{2} + \text{comisión}\right)$$

Parámetros usados:
- Spread = 0.1% → típico para ETFs líquidos (SPY, TLT, GLD)
- Comisión = 0.05% → tarifa de IBKR para ETFs (~$0.005/acción, muy barata)

**Turnover** mensual típico del TSMOM: ~20-40% del portfolio. Eso cuesta ~0.04%-0.08% por mes = ~0.5%-1% anual. **La diferencia bruto vs. neto es significativa.**

### 5.2 `src/backtest/metrics.py` — Métricas de performance

| Función | Qué mide |
|---------|----------|
| `sharpe_ratio(r)` | Retorno excedente / vol × √12. El benchmark estándar |
| `max_drawdown(r)` | Peor caída desde un máximo histórico. Mide el dolor |
| `calmar_ratio(r)` | Ann. return / |Max DD|. Retorno por unidad de drawdown |
| `sortino_ratio(r)` | Como Sharpe pero solo penaliza volatilidad a la baja |
| `turnover(w)` | Cambio promedio mensual de pesos → proxy del costo |
| `hit_rate(r)` | % de meses con retorno positivo |
| `performance_report(r, w)` | Dict con todas las métricas en formato printeable |

### 5.3 `src/backtest/engine.py` — BacktestEngine

**La clase `BacktestEngine` hace una sola cosa:** dado un DataFrame de retornos y un config, simula el portfolio mes a mes y devuelve un `BacktestResults`.

**Loop interno:**
```python
# (simplificado)
weights, signal = build_portfolio(returns, config)  # señales y pesos con .shift(1)
gross_returns   = (weights * returns).sum(axis=1)   # retorno bruto
costs           = compute_transaction_costs(weights) # costos de transacción
net_returns     = gross_returns - costs              # retorno neto
```

**`BacktestResults`** es un `dataclass` que calcula las métricas automáticamente al instanciarse (`__post_init__`). Su propiedad `cumulative` devuelve la equity curve.

In [ ]:
from src.backtest.engine import BacktestEngine
from src.backtest.metrics import print_report, annualized_return

engine = BacktestEngine(config)
results = engine.run(returns)

print_report(results.metrics, title='TSMOM — Señal Binaria')

# Impacto de los costos
ann_gross = annualized_return(results.gross_returns)
ann_net   = annualized_return(results.portfolio_returns)
print(f'Retorno bruto  : {ann_gross:.2%}')
print(f'Retorno neto   : {ann_net:.2%}')
print(f'Drag por costos: {ann_gross - ann_net:.2%} anual')

# Equity curve + drawdown
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

cum_net   = results.cumulative
cum_gross = (1 + results.gross_returns).cumprod()
axes[0].plot(cum_net.index,   cum_net.values,   label='Neto',   color='navy',     linewidth=1.8)
axes[0].plot(cum_gross.index, cum_gross.values, label='Bruto',  color='steelblue', linewidth=1, linestyle='--', alpha=0.6)
axes[0].set_yscale('log')
axes[0].set_title('Curva de Capital — Time Series Momentum (66 ETFs)')
axes[0].set_ylabel('Valor (log, base 1)')
axes[0].legend()

dd = (cum_net - cum_net.cummax()) / cum_net.cummax()
axes[1].fill_between(dd.index, dd.values * 100, 0, color='salmon', alpha=0.65)
axes[1].set_title('Drawdown (%)')
axes[1].set_ylabel('%')

plt.tight_layout(); plt.show()

In [ ]:
# Heatmap de retornos mensuales
import numpy as np

r = results.portfolio_returns
monthly_df = pd.DataFrame({'year': r.index.year, 'month': r.index.month, 'ret': r.values})
pivot = monthly_df.pivot(index='year', columns='month', values='ret') * 100
pivot.columns = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
pivot['Anual'] = (monthly_df.groupby('year')['ret']
                  .apply(lambda x: (1+x).prod() - 1) * 100)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-8, vmax=8)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=7.5)
plt.colorbar(im, ax=ax, label='Retorno (%)')
ax.set_title('Retornos Mensuales del Portfolio TSMOM (%)')
plt.tight_layout(); plt.show()

---
<a id='6'></a>
## 6. Tests de Robustez: `src/robustness/`

Un buen backtest no alcanza. Hay que demostrar que la estrategia **no es un artefacto de los parámetros elegidos**. Tres tests independientes:

### 6.1 `src/robustness/sensitivity.py` — Análisis de sensibilidad

Responde: *"Si cambio el lookback de 12 a 9 meses, ¿la estrategia colapsa?"*

- **`lookback_sensitivity()`**: corre el backtest para lookbacks ∈ {3, 6, 9, 12, 18, 24} meses
- **`target_vol_sensitivity()`**: idem para target_vol ∈ {5%, 8%, 10%, 12%, 15%, 20%}
- **`cost_sensitivity()`**: idem para spreads ∈ {0%, 0.1%, 0.5%, 1%, 2%}

**Interpretación:** si el Sharpe es positivo en toda la grilla de parámetros → la estrategia es robusta, no sobreajustada.

In [ ]:
from src.robustness.sensitivity import lookback_sensitivity, target_vol_sensitivity, cost_sensitivity

lb_df = lookback_sensitivity(returns, config)
tv_df = target_vol_sensitivity(returns, config)
cs_df = cost_sensitivity(returns, config)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Lookback
colors_lb = ['steelblue' if v > 0 else 'salmon' for v in lb_df['sharpe']]
axes[0].bar(lb_df.index, lb_df['sharpe'], color=colors_lb, alpha=0.85)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Sharpe por Lookback (meses)')
axes[0].set_xlabel('Lookback (meses)')

# Target vol
axes[1].plot([f'{v:.0%}' for v in tv_df.index], tv_df['sharpe'], 'o-', color='steelblue')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Sharpe por Target Volatility')
axes[1].set_xlabel('Target Vol')

# Costs
axes[2].plot([f'{v:.1%}' for v in cs_df.index], cs_df['sharpe'], 's-', color='darkorange')
axes[2].axhline(0, color='red', linewidth=0.8, linestyle='--', alpha=0.7)
axes[2].set_title('Sharpe por Spread de Costos')
axes[2].set_xlabel('Spread bid-ask')

for ax in axes:
    ax.set_ylabel('Sharpe Ratio')
    ax.grid(True, alpha=0.3)

plt.suptitle('Análisis de Sensibilidad de Parámetros', fontsize=13)
plt.tight_layout(); plt.show()

print('Sensibilidad al lookback:')
print(lb_df.round(2).to_string())

### 6.2 `src/robustness/stress_test.py` — Tests de estrés

Evalúa la performance durante los **peores períodos de mercado**.

**¿Por qué el TSMOM debería sobrevivir crisis?**
- En un crash sostenido (GFC 2008): después de 1-2 meses la señal se pone en -1 → el portfolio va short y **se beneficia** de la caída continuada
- El riesgo real son los *reversals* rápidos: COVID marzo-mayo 2020, donde el crash y la recuperación ocurrieron en solo 2 meses — el sistema no tuvo tiempo de girar
- 2022 (rate shock): caída simultánea de acciones y bonos → el TSMOM short en ambos puede funcionar bien

**`worst_drawdowns()`** identifica los 5 peores drawdowns históricos con su fecha de inicio, nadir y recuperación.

In [ ]:
from src.robustness.stress_test import analyze_crisis_performance, worst_drawdowns, CRISIS_PERIODS

crisis_df = analyze_crisis_performance(results.portfolio_returns)
print('Performance durante períodos de crisis:')
print(crisis_df.to_string())

print('\nPeores drawdowns históricos:')
print(worst_drawdowns(results.portfolio_returns, n=5).to_string(index=False))

# Equity curve con períodos de crisis sombreados
cum = results.cumulative
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(cum.index, cum.values, color='navy', linewidth=1.5)
colors_c = ['#ffcccc', '#ffe0b2', '#e8d5f5', '#c8e6c9']
for (name, (s, e)), col in zip(CRISIS_PERIODS.items(), colors_c):
    s_ts = pd.Timestamp(s); e_ts = pd.Timestamp(e)
    if s_ts <= cum.index[-1]:
        ax.axvspan(s_ts, min(e_ts, cum.index[-1]), alpha=0.5, color=col, label=name)
ax.set_title('Equity Curve con Períodos de Crisis')
ax.set_yscale('log')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 6.3 `src/robustness/out_of_sample.py` — Evaluación Out-of-Sample

**El problema del overfitting:** si optimizamos parámetros sobre todo el dataset, los resultados están sobreajustados. OOS evalúa si la estrategia **generaliza**.

**Walk-forward (`walk_forward()`):**
```
Ventana 1: train 2015-2019 (60m), test 2020 (12m)
Ventana 2: train 2015-2020 (72m), test 2021 (12m)
Ventana 3: train 2015-2021 (84m), test 2022 (12m)
...
```

**Expanding window (`expanding_window()`):**
Usa toda la historia disponible hasta cada punto, luego testea el mes siguiente.
Equivale a simular un trader real que va acumulando historia.

**Lo que buscamos:** Sharpe OOS ≈ Sharpe in-sample → no hay data-mining.

In [ ]:
from src.robustness.out_of_sample import walk_forward, expanding_window

wf_df = walk_forward(returns, config, train_years=5, test_years=1)
oos_summary = expanding_window(returns, config, min_train_months=36)

print('Walk-Forward Analysis (Sharpe OOS por ventana):')
print(wf_df[['test_start', 'test_end', 'sharpe', 'ann_ret', 'max_dd']].round(2).to_string(index=False))

print('\nIn-Sample vs Out-of-Sample:')
print(oos_summary.round(2).to_string())

if not wf_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    colors_wf = ['steelblue' if v >= 0 else 'salmon' for v in wf_df['sharpe']]
    ax.bar(wf_df['test_start'], wf_df['sharpe'], color=colors_wf, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('Sharpe Out-of-Sample por Ventana (Walk-Forward)')
    ax.set_xlabel('Inicio del período de test')
    ax.set_ylabel('Sharpe Ratio')
    plt.tight_layout(); plt.show()

---
<a id='7'></a>
## 7. Broker IBKR: `src/broker/`

El módulo broker conecta el sistema de trading a una cuenta real (o paper) de Interactive Brokers vía la biblioteca `ib_insync`, que abstrae la API oficial de IBKR.

### Arquitectura de comunicación

```
TWS (Trader Workstation, corriendo en tu PC)
       ↑↓  TCP socket  (puerto 7497 = paper, 7496 = real)
ib_insync  (biblioteca Python que implementa el protocolo)
       ↑↓
src/broker/ibkr.py     → IBKRConnection: conecta y lee datos de mercado
src/broker/orders.py   → execute_rebalance(): calcula y envía órdenes
src/broker/monitor.py  → stream_portfolio_updates(): loop de monitoreo
```

### `src/broker/ibkr.py` — `IBKRConnection`

```python
class IBKRConnection:
    connect(host, port, client_id)  # abre la conexión TCP con TWS
    get_account_summary()           # → {NetLiquidation, TotalCashValue, UnrealizedPnL}
    get_positions()                 # → DataFrame con ticker, qty, avg_cost, market_value
    get_market_price(ticker)        # → float con el último precio
    disconnect()
```

### `src/broker/orders.py` — `execute_rebalance()`

Recibe los `target_weights` calculados por la estrategia y el capital disponible, y:
1. Consulta las posiciones actuales en IBKR
2. Calcula la diferencia: `Δ_valor = w_target × capital − posición_actual`
3. Convierte en cantidad de acciones: `qty = Δ_valor / precio_mercado`
4. Descarta trades menores de 1 acción (para no fragmentar en fracciones)
5. Envía `MarketOrder` a TWS y registra en `logs/trades.csv`

**`--dry-run`**: muestra las órdenes sin ejecutarlas. Útil para verificar antes de operar.

### `src/broker/monitor.py` — `stream_portfolio_updates()`

Loop que cada 30 segundos:
1. Lee posiciones de IBKR
2. Calcula P&L no realizado
3. Imprime tabla formateada en la terminal
4. Alimenta la **Página 4** del dashboard Streamlit

### Setup para conectar a IBKR TWS (Paper Trading)

```
1. Abrir TWS → Edit → Global Configuration → API → Settings
2. ✅ Enable ActiveX and Socket Clients
3. Puerto: 7497 (paper) — NO usar 7496 (real) para el TP
4. Agregar 127.0.0.1 a Trusted IPs
5. OK → reiniciar TWS si pide
6. Ejecutar: python main.py monitor
```

In [ ]:
# Demo de la estructura del módulo broker (sin conectar a TWS)
# Para ver el código completo, abrir src/broker/ibkr.py

import inspect
from src.broker.ibkr import IBKRConnection
from src.broker.orders import execute_rebalance
from src.broker.monitor import print_portfolio_table

print('── IBKRConnection — métodos públicos ──')
methods = [m for m in dir(IBKRConnection) if not m.startswith('_')]
for m in methods:
    sig = inspect.signature(getattr(IBKRConnection, m))
    print(f'  .{m}{sig}')

print('\n── execute_rebalance — firma ──')
print(f'  {inspect.signature(execute_rebalance)}')

print('\n── print_portfolio_table — firma ──')
print(f'  {inspect.signature(print_portfolio_table)}')

In [ ]:
# Demo de dry-run: qué órdenes generaría el rebalanceo actual
# (sin conectar a IBKR — usa capital simulado de $1M)

from src.strategy.portfolio import build_portfolio

target_weights, _ = build_portfolio(returns, config)
latest_weights = target_weights.iloc[-1].dropna()
latest_weights = latest_weights[latest_weights != 0]

capital = 1_000_000
prices_sim = {t: 100.0 for t in latest_weights.index}  # precio simulado $100

print(f'Mes: {target_weights.index[-1].strftime("%Y-%m")} | Capital: ${capital:,.0f}')
print(f'{"─"*60}')
print(f'  {"Ticker":<8} {"Peso":>8} {"Valor ($)":>12} {"Acciones":>10} {"Acción"}')
print(f'{"─"*60}')

for ticker, w in latest_weights.sort_values().items():
    valor = w * capital
    qty = abs(valor) / prices_sim.get(ticker, 100.0)
    action = 'BUY' if w > 0 else 'SELL'
    print(f'  {ticker:<8} {w:>7.2%} {valor:>12,.0f} {qty:>10.0f}  {action}')

print(f'{"─"*60}')
print(f'  Total posiciones: {len(latest_weights)} | '
      f'Long: {(latest_weights>0).sum()} | Short: {(latest_weights<0).sum()}')

---
<a id='8'></a>
## 8. CLI: `main.py`

La interfaz de línea de comandos está construida con **Click**, el estándar de Python para CLIs profesionales.

**¿Por qué Click y no argparse?**
- Decoradores limpios: `@cli.command()`, `@click.option()`
- `@click.pass_context` para compartir `config_path` entre todos los comandos
- Colores en terminal (`click.secho`), help automático, manejo de errores

### Estructura del CLI

```
main.py
├── cli()              # grupo principal, acepta --config
├── update-data        # descarga ETFs nuevos vía yfinance
├── backtest           # corre el backtest, acepta --lookback, --target-vol, --start, --end, --signal
├── robustness         # corre tests, acepta --test {all, sensitivity, stress, oos}
├── monitor            # conecta a IBKR y muestra portfolio, acepta --interval
├── execute            # rebalancea en IBKR, acepta --dry-run
└── dashboard          # lanza Streamlit, acepta --port
```

### Patrón: overrides sobre config.yaml

Los comandos **no hardcodean valores**: leen `config.yaml` primero y luego aplican los argumentos del CLI encima. Así, se puede correr `python main.py backtest` sin argumentos (usa el config) o `python main.py backtest --lookback 6` (override solo ese parámetro).

In [ ]:
# Ver el help del CLI
import subprocess, sys
result = subprocess.run([sys.executable, 'main.py', '--help'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Help del comando backtest
result = subprocess.run([sys.executable, 'main.py', 'backtest', '--help'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Correr el backtest desde el CLI (demostración real)
result = subprocess.run(
    [sys.executable, 'main.py', 'backtest',
     '--lookback', '12', '--signal', 'binary'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])

In [ ]:
# Dry-run del execute: ver qué órdenes se generarían
result = subprocess.run(
    [sys.executable, 'main.py', 'execute', '--dry-run'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])

---
<a id='9'></a>
## 9. Dashboard: `src/dashboard/app.py`

El dashboard está construido con **Streamlit**, un framework que convierte código Python en una aplicación web interactiva.

**Comando para lanzarlo:**
```bash
python main.py dashboard
# o directamente:
streamlit run src/dashboard/app.py
```

### Estructura de las 5 páginas

```
Sidebar (siempre visible)
├── Selección de página
└── Parámetros ajustables en tiempo real
    ├── Lookback (slider)
    ├── Target Vol (slider)
    ├── Tipo de señal (binary / strength)
    └── Fechas del backtest

Páginas
├── 1. Overview          → señales actuales de los 66 ETFs + métricas del sidebar
├── 2. Backtest          → equity curve (Plotly), drawdown, heatmap mensual, tabla de métricas
├── 3. Robustness        → 3 tabs: sensibilidad | stress test | walk-forward OOS
├── 4. Live Portfolio    → conexión IBKR, posiciones, P&L, pie chart por asset class
└── 5. Trade Log         → tabla filtrable de trades + gráfico de actividad mensual
```

### Decisiones técnicas del dashboard

| Decisión | Razón |
|----------|-------|
| `@st.cache_data` en `_load_returns` y `_run_backtest` | El backtest tarda ~2s — cachear evita re-calcularlo cada vez que el usuario mueve un slider |
| Parámetros del sidebar como overrides sobre config | El usuario puede explorar variantes sin tocar archivos |
| Plotly en lugar de matplotlib | Gráficos interactivos (zoom, hover, descarga) |
| Página 4 con botón "Conectar" | La conexión IBKR es costosa — no conectar automáticamente al cargar |
| `config_str = yaml.dump(config)` como key del cache | Streamlit necesita un parámetro hasheable; el string YAML es la forma más simple |

In [ ]:
# Verificar que el dashboard es importable sin errores
import ast, pathlib

dashboard_path = pathlib.Path('src/dashboard/app.py')
code = dashboard_path.read_text(encoding='utf-8')

try:
    ast.parse(code)
    print('✓ src/dashboard/app.py — sintaxis válida')
    print(f'  Líneas de código: {len(code.splitlines())}')
    # Contar páginas
    pages = [l for l in code.splitlines() if 'elif page ==' in l or 'if page ==' in l]
    print(f'  Páginas definidas: {len(pages)}')
    for p in pages:
        print(f'    {p.strip()}')
except SyntaxError as e:
    print(f'✗ Error de sintaxis: {e}')

In [ ]:
# Simular la Página 1 del dashboard (Overview) en el notebook
# (misma lógica que app.py pero con matplotlib en lugar de Streamlit)

from src.strategy.signals import compute_tsmom_signal

signal_latest = compute_tsmom_signal(returns, lookback=12).iloc[-1].dropna()
date_label = returns.index[-1].strftime('%Y-%m')

sig_df = pd.DataFrame({
    'ticker':     signal_latest.index,
    'signal':     signal_latest.values.astype(int),
    'asset_class': [ETF_UNIVERSE.get(t, 'other') for t in signal_latest.index]
})

# Heatmap de señales por asset class
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, ac in zip(axes, ASSET_CLASSES):
    ac_df = sig_df[sig_df['asset_class'] == ac]
    colors_s = ['#4caf50' if v == 1 else '#f44336' for v in ac_df['signal']]
    ax.barh(ac_df['ticker'], ac_df['signal'], color=colors_s, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{ac.capitalize()} ({len(ac_df)} ETFs)')
    ax.set_xlim(-1.5, 1.5)
    ax.set_xticks([-1, 0, 1])
    ax.set_xticklabels(['SHORT', '', 'LONG'])

plt.suptitle(f'Señales TSMOM — {date_label} (verde=LONG, rojo=SHORT)', fontsize=12)
plt.tight_layout(); plt.show()

n_long  = (signal_latest == 1).sum()
n_short = (signal_latest == -1).sum()
print(f'Posiciones LONG: {n_long} | SHORT: {n_short} | Net: {n_long - n_short:+d}')

---
<a id='10'></a>
## 10. Configuración Central: `config.yaml`

**Toda la parametrización del sistema vive en un único archivo.** Cambiar un valor aquí cambia el comportamiento de todo el sistema sin tocar el código.

```yaml
strategy:
  lookback_months: 12      # Ventana TSMOM (paper: 12 meses)
  target_volatility: 0.10  # Vol objetivo anual del portafolio (paper: 40% para futuros;
                           #   usamos 10% porque ETFs ya tienen apalancamiento bajo)
  max_position_weight: 0.15 # Cap por posición (evita concentración)
  vol_com_months: 3        # EWMA center-of-mass ≈ 60 días de trading

transaction_costs:
  bid_ask_spread: 0.001    # 10 bps — típico para ETFs con >$1B AUM
  commission_pct: 0.0005   # 5 bps — tarifa IBKR tiered pricing

backtest:
  start_date: '2015-01-01' # Requiere ≥12 meses de warmup después de 2014
  end_date: '2024-12-31'
  initial_capital: 1000000

data:
  path: 'Data/etf_prices.parquet'
  yfinance_update: false   # true = descarga precios nuevos al cargar

broker:
  host: '127.0.0.1'
  port: 7497               # 7497 = TWS paper, 7496 = TWS real, 4001 = IB Gateway paper
  client_id: 1
```

In [ ]:
# Mostrar el config actual
import yaml
with open('config.yaml') as f:
    config_loaded = yaml.safe_load(f)

print('config.yaml actual:')
print('─' * 40)
for section, values in config_loaded.items():
    print(f'[{section}]')
    if isinstance(values, dict):
        for k, v in values.items():
            print(f'  {k}: {v}')
    else:
        print(f'  {values}')
    print()

---
<a id='11'></a>
## 11. Demo End-to-End

Esta sección demuestra el sistema completo en acción:
1. Cargar datos → 2. Correr backtest → 3. Robustez → 4. Generar órdenes → 5. Comparar configuraciones

In [ ]:
# ── 1. Pipeline completo de punta a punta ──
import yaml
from src.data.loader import load_returns
from src.backtest.engine import BacktestEngine
from src.backtest.metrics import print_report

with open('config.yaml') as f:
    config = yaml.safe_load(f)

returns = load_returns(config)
engine  = BacktestEngine(config)
results = engine.run(returns)

print_report(results.metrics, title='TSMOM — Señal Binaria (lookback=12m)')

In [ ]:
# ── 2. Comparación: señal binaria vs señal continua ──
from copy import deepcopy
from src.backtest.metrics import sharpe_ratio, annualized_return, max_drawdown

results_bin  = BacktestEngine(config).run(returns, use_signal_strength=False)
results_cont = BacktestEngine(config).run(returns, use_signal_strength=True)

comparison = pd.DataFrame({
    'Señal Binaria (Moskowitz)': [
        sharpe_ratio(results_bin.portfolio_returns),
        annualized_return(results_bin.portfolio_returns),
        max_drawdown(results_bin.portfolio_returns),
    ],
    'Señal Continua (Baz)': [
        sharpe_ratio(results_cont.portfolio_returns),
        annualized_return(results_cont.portfolio_returns),
        max_drawdown(results_cont.portfolio_returns),
    ],
}, index=['Sharpe Ratio', 'Ann. Return', 'Max Drawdown'])

print('Comparación de señales:')
print(comparison.applymap(lambda x: f'{x:.2f}' if abs(x) > 1 else f'{x:.2%}'))

In [ ]:
# ── 3. Equity curves comparadas ──
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(results_bin.cumulative.index, results_bin.cumulative.values,
        label='Binaria (Moskowitz 2012)', color='navy', linewidth=1.8)
ax.plot(results_cont.cumulative.index, results_cont.cumulative.values,
        label='Continua (Baz et al. 2015)', color='darkorange', linewidth=1.8, linestyle='--')

ax.set_yscale('log')
ax.set_title('Comparación de Señales — Binaria vs Continua', fontsize=13)
ax.set_ylabel('Valor del portafolio (log, base 1)')
ax.legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── 4. Resumen de la estructura de archivos del proyecto ──
import pathlib

root = pathlib.Path('.')
print('Estructura del proyecto:')
for path in sorted(root.rglob('*.py')):
    parts = path.parts
    if any(p in parts for p in ['.git', '__pycache__', '.ipynb_checkpoints']):
        continue
    indent = '  ' * (len(parts) - 1)
    lines = len(path.read_text(encoding='utf-8', errors='ignore').splitlines())
    print(f'{indent}{path.name:35} ({lines:4d} líneas)')

print()
# Contar también archivos de config y datos
for fname in ['config.yaml', 'requirements.txt', 'Data/etf_prices.parquet']:
    p = pathlib.Path(fname)
    if p.exists():
        size = p.stat().st_size
        print(f'  {fname:35} ({size/1024:.1f} KB)')

---

## Resumen de Decisiones de Diseño

| Decisión | Alternativa considerada | Por qué elegimos esto |
|----------|------------------------|----------------------|
| ETFs en lugar de futuros | Futuros (como el paper original) | Datos gratuitos (yfinance), ejecutables en IBKR paper sin margen para futuros |
| Señal binaria `sign()` como default | Señal continua de Baz | Más robusta, menor turnover, replica exactamente el paper principal |
| Rebalanceo mensual | Semanal o diario | Menores costos (turnover 4× menor), consistente con el paper |
| EWMA com=3m para vol | Ventana fija de 12m | Reacciona más rápido a cambios de régimen (Barroso & SC 2015 lo valida) |
| `target_vol = 10%` anual | Notional fijo | Comparabilidad entre activos, gestión de riesgo explícita |
| Streamlit para dashboard | Dash / Flask / tkinter | Una línea para lanzar, código Python puro, gráficos Plotly interactivos |
| `ib_insync` para IBKR | `ibapi` oficial de IBKR | API de más alto nivel, async built-in, documentación mucho mejor |
| `config.yaml` centralizado | Parámetros en cada módulo | Un solo lugar para cambiar parámetros → reproducibilidad garantizada |
| `BacktestResults` como dataclass | Dict o tuple | Las métricas se calculan automáticamente en `__post_init__` → no hay que recordar llamarlas |
| `.shift(1)` en vol y señal | Calcular en $t$ | Garantiza **cero look-ahead bias** en todas las señales |

---

## Cómo correr el sistema

```bash
# Instalar dependencias
pip install -r requirements.txt

# Backtest completo
python main.py backtest

# Backtest con parámetros custom
python main.py backtest --lookback 9 --target-vol 0.12 --signal strength

# Todos los tests de robustez
python main.py robustness --test all

# Dashboard interactivo
python main.py dashboard

# Ver órdenes del próximo rebalanceo (sin ejecutar)
python main.py execute --dry-run

# Monitorear portfolio en IBKR (requiere TWS abierto)
python main.py monitor

# Actualizar datos de mercado
python main.py update-data
```